In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
df = pd.read_csv("IPL_Stats.csv")

C:\Users\asus\AppData\Local\Temp\ipykernel_7048\1759675776.py:1: DtypeWarning: Columns (0: Season) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("IPL_Stats.csv")


In [4]:
print("--- 1. Understanding the Dataset ---")
print("Shape of dataset:", df.shape)
print("Columns:", df.columns.tolist())

print("\n--- 2. Checking & Handling Missing Values ---")
df['type of extras'] = df['type of extras'].fillna('No Extra')
df['wicket_type'] = df['wicket_type'].fillna('No Wicket')
df['fielders_involved'] = df['fielders_involved'].fillna('None')
df['Player Out'] = df['Player Out'].fillna('None')

print("\n--- 3. Removing Duplicate Records ---")
df = df.drop_duplicates()

print("\n--- 8. Removing Irrelevant or Redundant Features ---")
columns_to_drop = [
    'Match id', 'Date', 'score/wicket', 'Striker', 'Non Striker', 
    'Bowler', 'Player Out', 'fielders_involved', 'type of extras', 'wicket_type'
]
df = df.drop(columns=columns_to_drop)

print("\n--- 4. Detecting & Handling Outliers ---")
Q1 = df['extras'].quantile(0.25)
Q3 = df['extras'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
df['extras'] = np.where(df['extras'] > upper_bound, upper_bound, df['extras'])

print("\n--- 5. Handling Skewness ---")
df['runs_scored_sqrt'] = np.sqrt(df['runs_scored'])
df = df.drop(columns=['runs_scored'])

X = df.drop(columns=['score'])
y = df['score']

print("\n--- 6. Handling Categorical Variables ---")
X = pd.get_dummies(X, columns=['Season', 'Batting team', 'Bowling team'], drop_first=True)

--- 1. Understanding the Dataset ---
Shape of dataset: (255759, 19)
Columns: ['Match id', 'Date', 'Season', 'Batting team', 'Bowling team', 'Innings No', 'Ball No', 'Bowler', 'Striker', 'Non Striker', 'runs_scored', 'extras', 'type of extras', 'score', 'score/wicket', 'wicket_confirmation', 'wicket_type', 'fielders_involved', 'Player Out']

--- 2. Checking & Handling Missing Values ---

--- 3. Removing Duplicate Records ---

--- 8. Removing Irrelevant or Redundant Features ---

--- 4. Detecting & Handling Outliers ---

--- 5. Handling Skewness ---

--- 6. Handling Categorical Variables ---


In [5]:
X_sampled, _, y_sampled, _ = train_test_split(X, y, train_size=20000, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_sampled, y_sampled, test_size=0.2, random_state=42)

print("\n--- 7. Feature Scaling ---")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Preprocessing Complete! Ready for modeling.\n")

print("Training 3 Regression Algorithms...")

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42, max_depth=10),
    "Random Forest": RandomForestRegressor(random_state=42, n_estimators=50, max_depth=10)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    
    y_pred = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = [mae, mse, rmse, r2]

metrics_df = pd.DataFrame(
    results, 
    index=["MAE", "MSE", "RMSE", "R-Squared"]
).T 

print("\n--- Regression Metrics Comparison Matrix ---")
print(metrics_df.round(4))


--- 7. Feature Scaling ---
Preprocessing Complete! Ready for modeling.

Training 3 Regression Algorithms...

--- Regression Metrics Comparison Matrix ---
                       MAE       MSE     RMSE  R-Squared
Linear Regression  12.0884  269.7706  16.4247     0.8882
Decision Tree      12.4118  309.0105  17.5787     0.8719
Random Forest      11.7487  267.2361  16.3474     0.8892
